In [ ]:
# Colab / fresh environment: install the module (builds the C++ core, takes a few minutes the first time)
try:
    import sensor_fusion
except ImportError:
    %pip install -q git+https://github.com/Rudip1/learn-sensor-fusion

# 2 · Coordinate frames: WGS-84, ECEF, ENU and UTM

**Recap** ([theory](../../1_theory/02_coordinate_frames.md)). WGS-84 is an ellipsoid with $a$ and $f$
(2.1). Geodetic $(\varphi,\lambda,h)$ goes to ECEF in closed form with the prime-vertical radius
$N(\varphi)$ (2.2)–(2.3); the way back is a short fixed-point iteration (Algorithm 2.1). A local ENU frame
is ECEF rotated by $\mathbf R_{EN}$ (2.4) and shifted to an origin (2.5): exact, but a plane. The
flat-Earth shortcut (2.7) is exact to first order; "111 km per degree" is not. UTM (2.8)–(2.12) is a
conformal projection with a scale $k\neq 1$ and a grid convergence $\gamma$.

You will implement (2.3) and (2.4) yourself, check the C++ code against PROJ, put the recorded tracks into
ENU and UTM, and measure what the shortcuts cost.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import sensor_fusion as sf
from sensor_fusion import geo, nmea
from sensor_fusion.plotting import SERIES, use_style

use_style()
WGS84 = geo.WGS84
print(f"a = {WGS84.a} m, 1/f = {1 / WGS84.f:.9f}, b = {WGS84.b:.4f} m, e^2 = {WGS84.e2:.12f}")

## 1. Geodetic → ECEF

### ✏️ Exercise 1
Implement eq. (2.3) for arrays of latitude/longitude in degrees and heights in metres. Return an
`(n, 3)` array.

In [ ]:
def my_geodetic_to_ecef(lat_deg, lon_deg, h):
    # ✏️ your code here
    raise NotImplementedError


In [ ]:
rng = np.random.default_rng(0)
lat = rng.uniform(-90, 90, 1000); lon = rng.uniform(-180, 180, 1000); h = rng.uniform(-100, 20_000, 1000)
mine = my_geodetic_to_ecef(lat, lon, h)
assert np.abs(mine - geo.geodetic_to_ecef(lat, lon, h)).max() < 1e-6
print("✅ matches the C++ conversion on 1000 random points")

### Reference check against PROJ
PROJ (through `pyproj`) is the standard open-source coordinate library. It is used here only as a
reference: EPSG:4979 is WGS-84 geodetic 3D, EPSG:4978 is WGS-84 ECEF.

In [ ]:
try:
    from pyproj import Transformer
    to_ecef = Transformer.from_crs("EPSG:4979", "EPSG:4978", always_xy=True)  # note: lon, lat order!
    ref = np.column_stack(to_ecef.transform(lon, lat, h))
    print(f"max |C++ - PROJ| ECEF: {np.abs(geo.geodetic_to_ecef(lat, lon, h) - ref).max() * 1e6:.3f} µm")
    back = geo.ecef_to_geodetic(ref)
    print(f"round trip: max |dlat| {np.abs(back[:, 0] - lat).max():.2e} deg, max |dh| {np.abs(back[:, 2] - h).max() * 1e6:.3f} µm")
except ImportError:
    print("pyproj not installed: pip install pyproj to run the reference comparison")

## 2. ENU at a local origin

### ✏️ Exercise 2
Build $\mathbf R_{EN}$ of eq. (2.4) (columns: east, north, up expressed in ECEF) and use it with (2.5) to
express a point 50 m higher than the origin in ENU. You should get $(0, 0, 50)$.

In [ ]:
lat0, lon0, h0 = 47.4724, 19.0633, 148.0

def my_rotation_ecef_enu(lat_deg, lon_deg):
    # ✏️ your code here
    raise NotImplementedError


R = my_rotation_ecef_enu(lat0, lon0)
origin = my_geodetic_to_ecef([lat0], [lon0], [h0])[0]
above = my_geodetic_to_ecef([lat0], [lon0], [h0 + 50])[0]
# ✏️ your code here
raise NotImplementedError
print("ENU of a point 50 m above the origin:", np.round(enu_above, 9))

In [ ]:
assert np.allclose(R, geo.rotation_ecef_enu(lat0, lon0), atol=1e-15)
assert np.allclose(R.T @ R, np.eye(3), atol=1e-14)
assert np.allclose(enu_above, [0, 0, 50], atol=1e-6)
print("✅ rotation and ENU conversion correct")

## 3. The recorded tracks in metres

The walk (`nmea_log2.nmea`, heights $h = H + N_g$) and the parking-lot drive (`run1_fix.csv`, a ROS
`NavSatFix` log whose altitude is already ellipsoidal), each in ENU at its own first fix.

In [ ]:
walk = sf.tables.epochs_to_arrays(nmea.parse_log_file(str(sf.data.path("gnss/nmea_log2.nmea"))).epochs)
walk_h = walk["alt_msl"] + walk["geoid_sep"]  # eq. (1.8)
ltp_walk = geo.LocalTangentPlane(walk["lat"][0], walk["lon"][0], walk_h[0])
walk_enu = ltp_walk.geodetic_to_enu(walk["lat"], walk["lon"], walk_h)

drive = sf.io.load_navsatfix("parking_lot/run1_fix.csv")
ltp_drive = geo.LocalTangentPlane(drive["lat"][0], drive["lon"][0], drive["alt"][0])
drive_enu = ltp_drive.geodetic_to_enu(drive["lat"], drive["lon"], drive["alt"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2), constrained_layout=True)
for ax, enu, title in [(axes[0], walk_enu, "walk (NMEA log 2)"), (axes[1], drive_enu, "parking lot, run 1")]:
    ax.plot(enu[:, 0], enu[:, 1], ".-", ms=2, lw=0.8, color=SERIES[0])
    ax.plot(*enu[0, :2], "o", color=SERIES[2], label="start")
    ax.plot(*enu[-1, :2], "s", color=SERIES[1], label="end")
    ax.set_aspect("equal"); ax.set_xlabel("east [m]"); ax.set_ylabel("north [m]"); ax.set_title(title)
axes[0].legend()
plt.show()
print(f"walk: extent {np.ptp(walk_enu[:, 0]):.0f} m E x {np.ptp(walk_enu[:, 1]):.0f} m N; "
      f"parking lot: {np.ptp(drive_enu[:, 0]):.1f} m E x {np.ptp(drive_enu[:, 1]):.1f} m N")

## 4. UTM and what it does to directions and distances

In [ ]:
u = geo.geodetic_to_utm(drive["lat"], drive["lon"])
print(f"zone {u['zone']}{'N' if u['north'] else 'S'}, first fix E = {u['easting'][0]:.3f} m, N = {u['northing'][0]:.3f} m")
print(f"scale k = {u['scale'][0]:.7f}, grid convergence = {u['convergence_deg'][0]:.4f} deg")

try:
    from pyproj import Transformer
    to_utm = Transformer.from_crs("EPSG:4326", f"EPSG:{32600 + u['zone']}", always_xy=True)
    e_ref, n_ref = to_utm.transform(drive["lon"], drive["lat"])
    print(f"max |C++ - PROJ| UTM: {max(np.abs(u['easting'] - e_ref).max(), np.abs(u['northing'] - n_ref).max()) * 1e6:.3f} µm")
except ImportError:
    pass

UTM coordinates relative to the first fix are the ENU coordinates *rotated by the grid convergence* and
*scaled by k* (to first order). Check it on the drive:

In [ ]:
du = np.column_stack([u["easting"] - u["easting"][0], u["northing"] - u["northing"][0]])
g = np.radians(u["convergence_deg"][0])
Rg = np.array([[np.cos(g), -np.sin(g)], [np.sin(g), np.cos(g)]])
predicted = u["scale"][0] * drive_enu[:, :2] @ Rg.T
print(f"max |UTM offset - k R(gamma) ENU| = {np.abs(du - predicted).max() * 1000:.2f} mm "
      f"(raw difference without the correction: {np.abs(du - drive_enu[:, :2]).max():.2f} m)")

## 5. 🔨 Break it

### 🔨 5.1 Lat/lon swapped
One of the most common bugs: `transform(lat, lon)` where the library expects `(lon, lat)`.

In [ ]:
wrong = geo.geodetic_to_ecef(drive["lon"][0], drive["lat"][0], drive["alt"][0])
right = geo.geodetic_to_ecef(drive["lat"][0], drive["lon"][0], drive["alt"][0])
print(f"swapping lat and lon moves the point by {np.linalg.norm(wrong - right) / 1000:.0f} km")

### 🔨 5.2 "111 km per degree" and forgetting cos(latitude)
The original analysis of the walk used $111\,000\cos\bar\varphi$ m per degree of longitude. Compare it,
and a version without the cosine, with the exact ENU track.

In [ ]:
k = 111_000.0
crude = np.column_stack([(walk["lon"] - walk["lon"][0]) * k * np.cos(np.radians(walk["lat"][0])),
                         (walk["lat"] - walk["lat"][0]) * k])
no_cos = np.column_stack([(walk["lon"] - walk["lon"][0]) * k, (walk["lat"] - walk["lat"][0]) * k])
for name, xy in [("111 km x cos(lat)", crude), ("111 km, no cos", no_cos)]:
    err = np.linalg.norm(xy - walk_enu[:, :2], axis=1)
    print(f"{name:18s}: max error {err.max():7.2f} m over a {np.ptp(walk_enu[:, 0]):.0f} m track")
plt.figure(figsize=(5, 4.2))
plt.plot(walk_enu[:, 0], walk_enu[:, 1], color=SERIES[0], label="exact ENU")
plt.plot(no_cos[:, 0], no_cos[:, 1], color=SERIES[3], label="no cos(lat)")
plt.gca().set_aspect("equal"); plt.legend(); plt.xlabel("east [m]"); plt.ylabel("north [m]"); plt.show()

### 🔨 5.3 One ENU frame for a whole country
ENU is exact but flat. Drive "level" (constant $h$) away from the origin and watch the up coordinate.

In [ ]:
ltp = geo.LocalTangentPlane(lat0, lon0, h0)
for km in [0.1, 1, 10, 50, 100]:
    p = ltp.enu_to_geodetic([km * 1000, 0, 0])  # a point on the plane, km east
    on_ground = ltp.geodetic_to_enu(p[0], p[1], h0)  # same lat/lon, but at the origin's height
    print(f"{km:6.1f} km east: level ground lies {-on_ground[2]:9.3f} m below the ENU plane")

### 🔨 5.4 Crossing a UTM zone boundary
Two points about 200 m apart on either side of 18° E (the boundary between zones 33 and 34), each projected into
its own zone, versus both projected into one zone.

In [ ]:
a = geo.geodetic_to_utm(47.47, 17.9987)
b = geo.geodetic_to_utm(47.47, 18.0013)
print(f"own zones {a['zone']} and {b['zone']}: easting difference {b['easting'] - a['easting']:.1f} m")
b33 = geo.geodetic_to_utm(47.47, 18.0013, zone=33)
print(f"both in zone 33: easting difference {b33['easting'] - a['easting']:.1f} m")

## What to remember

- Latitude is *geodetic* (along the ellipsoid normal); heights in conversions are ellipsoidal $h$.
- ECEF ↔ geodetic: closed form one way, a quickly converging iteration the other way.
- ENU = rotated and shifted ECEF: exact, but a plane — use it locally (a few km) as the world frame.
- "111 km per degree" has a first-order error; at least use $N(\varphi_0)\cos\varphi_0$ and $M(\varphi_0)$.
- UTM is conformal but scaled ($k$) and rotated ($\gamma$) with respect to ENU; keep a data set in one zone.